In [0]:
%pip install geopy
%pip install openmeteo-requests

In [0]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="friendly-air-quality-predictor")
location = geolocator.geocode("Lugano, Università Svizzera Italiana")
location

In [0]:
df = spark.sql("""
    SELECT 
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date
    FROM workspace.mlfsbook.lugano_aq_fg
""")

display(df)

row = df.collect()[0]
earliest_date = row['earliest_date']
latest_date = row['latest_date']

In [0]:
import openmeteo_requests

openmeteo = openmeteo_requests.Client()

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": location.latitude,
    "longitude": location.longitude,
    "start_date": earliest_date.strftime("%Y-%m-%d"),
    "end_date": latest_date.strftime("%Y-%m-%d"),
    "daily": ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max", "wind_direction_10m_dominant"],
    "timezone": "Europe/Zurich"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

In [0]:
import pandas as pd

daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(1).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(2).ValuesAsNumpy()
daily_wind_direction_10m_dominant = daily.Variables(3).ValuesAsNumpy()
daily_data = {"date": pd.date_range(
    start = pd.to_datetime(daily.Time(), unit = "s"),
    end = pd.to_datetime(daily.TimeEnd(), unit = "s"),
    freq = pd.Timedelta(seconds = daily.Interval()),
    inclusive = "left"
)}
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["wind_direction_10m_dominant"] = daily_wind_direction_10m_dominant

df_daily_weather_archive = pd.DataFrame(data = daily_data)
df_daily_weather_archive = df_daily_weather_archive.dropna()
df_daily_weather_archive['city'] = "Lugano"

display(df_daily_weather_archive)

In [0]:
url = "https://api.open-meteo.com/v1/ecmwf"
params = {
    "latitude": location.latitude,
    "longitude": location.longitude,
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m", "wind_direction_10m"]
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]

# Process hourly data. The order of variables needs to be the same as requested.

hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
    start = pd.to_datetime(hourly.Time(), unit = "s"),
    end = pd.to_datetime(hourly.TimeEnd(), unit = "s"),
    freq = pd.Timedelta(seconds = hourly.Interval()),
    inclusive = "left"
)}
hourly_data["temperature_2m_mean"] = hourly_temperature_2m
hourly_data["precipitation_sum"] = hourly_precipitation
hourly_data["wind_speed_10m_max"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m_dominant"] = hourly_wind_direction_10m

df_weather_hourly_forecast = pd.DataFrame(data = hourly_data)
df_weather_hourly_forecast = df_weather_hourly_forecast.dropna()

display(df_weather_hourly_forecast)

In [0]:
df_weather_fg = pd.concat([df_daily_weather_archive, df_weather_hourly_forecast])

display(df_weather_fg)

In [0]:
spark_df = spark.createDataFrame(df_weather_fg)
spark_df.write.saveAsTable("workspace.mlfsbook.lugano_weather_fg")
